# Component I. Vector Indexing and Embedding

# DecisionThree: JRU Program Inquiry Chatbot

**Group:** DecisionThree  
**Member:** Job Russel Enriquez  
**My dataset:** `IT_and_Business_Analytics.pdf`


Our group is building a chatbot that answers questions about José Rizal University college programs using information from JRU documents. We divided the dataset into three documents, with one document assigned to each member.

My document covers BS Information Technology (BSIT) and BSIT with Specialization in Business Analytics (BSIT-BA). This notebook reads the PDF, divides its text into smaller parts, and makes those parts searchable. The chatbot then uses the relevant parts to answer questions about these two programs.

### 1. INSTALL REQUIRED LIBRARIES


### Cell 1: Install Required Libraries

This cell's purpose is to install the Python packages used by the indexing and retrieval workflow. Without these dependencies, the notebook cannot read PDFs, split and embed their text, build a FAISS index, or export a review workbook. Colab sessions are temporary, so this installation step may be needed again after a runtime reset.

`!pip install -q` installs packages with minimal output; `faiss-cpu` provides the vector similarity search used later in the notebook.

`sentence-transformers` loads the embedding model, while `pypdf` extracts text from the curriculum PDF.

`transformers` loads Qwen and its tokenizer, while `accelerate` helps place the model on the available device. The same installation command also includes `bitsandbytes` for 4-bit weights and `sentencepiece` for tokenizer support.

`pandas` organizes the chunk records into a table, and `openpyxl` writes and formats the Excel inspection workbook.



In [1]:
!pip install -q \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    "transformers>=4.51.0" \
    accelerate \
    "bitsandbytes>=0.46.1" \
    sentencepiece \
    pandas \
    openpyxl

### 2. IMPORT LIBRARIES



### Cell 2: Import Libraries

This cell's purpose is to import the libraries needed by the first component of the notebook. Keeping imports together makes dependencies visible and ensures the later PDF, chunking, embedding, indexing, and export cells have their tools available. The imports are grouped here so a missing package is discovered before the main workflow starts.

`from google.colab import files` and `from pypdf import PdfReader` are used to import Google Colab's upload helper, PdfReader, and SentenceTransformer.

`import faiss` and `import numpy as np` are used to import FAISS, NumPy, pandas, pickle, regular expressions, and operating-system path utilities for indexing and file handling.

`from openpyxl.styles import Font, PatternFill, Alignment` is used to import Excel formatting helpers used to make the exported chunk inspection sheet readable.



In [2]:
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np
import pandas as pd
import pickle
import re
import os

from openpyxl.styles import Font, PatternFill, Alignment

### 3. INDEXING PARAMETERS



### Cell 3: Indexing Parameters

This cell's purpose is to set the experimental chunking and retrieval parameters. These settings control how the source document is divided and how many candidate chunks retrieval returns; the embedding model is held constant for fair comparisons. With the current settings, a new 1,500-character chunk begins every 1,200 characters because 300 characters overlap.

`CHUNK_SIZE = 1500` and `CHUNK_OVERLAP = 300` are used to define the chunk size and overlap. Adjust these values when testing chunking configurations.

`TOP_K = 3` is used to set the default number of retrieved chunks, Top K, to three.

`EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"` is used to name the fixed embedding model used to represent both document chunks and questions.



In [3]:
# Change these values to experiment with chunking.

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 300
TOP_K = 3

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

### 4. VALIDATE CHUNK PARAMETERS



### Cell 4: Validate Chunk Parameters

This cell's purpose is to validate the experiment settings before processing the PDF. Early checks catch invalid parameter combinations that could create empty chunks, non-terminating loops, or an invalid retrieval request. In particular, an overlap equal to the chunk size would make the chunking loop stop advancing.

`if CHUNK_SIZE <= 0:` and `raise ValueError("CHUNK_SIZE must be greater than 0.")` are used to reject a chunk size of zero or less.

`if CHUNK_OVERLAP < 0:` and `raise ValueError("CHUNK_OVERLAP cannot be negative.")` are used to reject negative overlap.

`if CHUNK_OVERLAP >= CHUNK_SIZE:` is used to ensure overlap is smaller than chunk size so each chunking step advances.

`if TOP_K <= 0:` and `raise ValueError("TOP_K must be greater than 0.")` are used to require a positive Top K.

`print("RAG INDEXING CONFIGURATION")` and `print("Chunk size       :", CHUNK_SIZE)` are used to print the active values so the run can be checked and documented.



In [4]:
if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be greater than 0.")

if CHUNK_OVERLAP < 0:
    raise ValueError("CHUNK_OVERLAP cannot be negative.")

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError(
        "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
    )

if TOP_K <= 0:
    raise ValueError("TOP_K must be greater than 0.")


print("=" * 70)
print("RAG INDEXING CONFIGURATION")
print("=" * 70)

print("Chunk size       :", CHUNK_SIZE)
print("Chunk overlap    :", CHUNK_OVERLAP)
print("Top-K             :", TOP_K)
print("Embedding model   :", EMBEDDING_MODEL_NAME)

RAG INDEXING CONFIGURATION
Chunk size       : 1500
Chunk overlap    : 300
Top-K             : 3
Embedding model   : all-MiniLM-L6-v2


### 5. UPLOAD PDF DOCUMENT



### Cell 5: Upload PDF Document

This cell's purpose is to upload and validate the source PDF in Google Colab. This connects the experiment to the chosen curriculum dataset and prevents later cells from silently processing no file or the wrong file type. This cell uses only the first uploaded file, so the chosen PDF should be uploaded by itself.

`print("UPLOAD DOCUMENT")` is used to print a heading for the upload stage.

`uploaded = files.upload()` is used to open Colab's file picker and collect uploaded files.

`if len(uploaded) == 0:` and `raise RuntimeError("No file was uploaded.")` are used to stop if the user did not upload anything.

`pdf_file = next(iter(uploaded.keys()))` is used to select the first uploaded filename for this run.

`if not pdf_file.lower().endswith(".pdf"):` is used to check that the selected filename ends in .pdf and explain the expected input if it does not.

`print("\nSelected file:")` and `print(pdf_file)` are used to display the chosen filename for confirmation.



In [5]:
print("\n")
print("=" * 70)
print("UPLOAD DOCUMENT")
print("=" * 70)

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No file was uploaded.")

# Use the first uploaded PDF
pdf_file = next(iter(uploaded.keys()))

if not pdf_file.lower().endswith(".pdf"):
    raise ValueError(
        "The uploaded file is not a PDF. Please upload a PDF."
    )

print("\nSelected file:")
print(pdf_file)



UPLOAD DOCUMENT


Saving IT_and_Business_Analytics.pdf to IT_and_Business_Analytics (1).pdf

Selected file:
IT_and_Business_Analytics (1).pdf


### 6. EXTRACT TEXT FROM PDF



### Cell 6: Extract Text From PDF

This cell's purpose is to extract and clean text from every page of the uploaded PDF. RAG can retrieve only from text that was successfully extracted; preserving page numbers also lets answers report where evidence came from. A page with no extractable text is not added to the searchable dataset.

`print("EXTRACTING PDF TEXT")` is used to label the extraction stage in the output.

`reader = PdfReader(pdf_file)` is used to open the selected PDF with pypdf.

`pages = []` is used to create a list that will hold non-empty page records.

`for page_number, page in enumerate(reader.pages, start=1):` and `page_text = page.extract_text()` are used to visit pages in order, starting page numbering at one, and extract each page's text.

`if page_text:` and `page_text` are used to skip empty pages and normalize repeated whitespace into a cleaner single-space text string.

`if page_text:` and `pages.append({` are used to store each usable page as a record containing its page number and cleaned text.

`print("PDF pages:", len(reader.pages))` and `print("Pages containing text:", len(pages))` are used to report total pages and pages with extractable text to reveal scanning or extraction issues.



In [6]:
print("\n")
print("=" * 70)
print("EXTRACTING PDF TEXT")
print("=" * 70)

reader = PdfReader(pdf_file)

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if page_text:

        # Normalize whitespace
        page_text = re.sub(
            r"\s+",
            " ",
            page_text
        ).strip()

        if page_text:
            pages.append({
                "page": page_number,
                "text": page_text
            })


print("PDF pages:", len(reader.pages))
print("Pages containing text:", len(pages))



EXTRACTING PDF TEXT
PDF pages: 22
Pages containing text: 22


### 7. CHECK WHETHER TEXT WAS EXTRACTED



### Cell 7: Check Whether Text Was Extracted

This cell's purpose is to stop when PDF text extraction produced no usable pages. This avoids creating an empty index and tells the user when OCR may be needed for a scanned image-only PDF. If the PDF contains only page images, optical character recognition is needed before this notebook can index its content.

`if len(pages) == 0:` is used to check the extracted page list; raise a clear error if it is empty, including the likely scanned-PDF cause.



In [7]:
if len(pages) == 0:
    raise RuntimeError(
        "No text could be extracted from the PDF. "
        "The PDF may consist of scanned images and require OCR."
    )

### 8. CREATE DOCUMENT CHUNKS



### Cell 8: Create Document Chunks

This cell's purpose is to define and run the character-based chunking function. Chunking turns long pages into smaller retrieval units while keeping source page numbers. Overlap reduces the chance that an answer is split across a chunk boundary. Chunk IDs increase in order, and each chunk keeps the page number of the page from which it was created.

`create_chunks` is used to define the function inputs and document that chunks are created separately for each page, with overlap within that page.

`chunks = []` and `chunk_id = 0` are used to initialize the output list and sequential chunk IDs; compute the forward step as chunk size minus overlap.

`for page_data in pages:` and `page_number = page_data["page"]` are used to process each page record and keep its page number and text together.

`start = 0` and `while start < len(page_text):` are used to move through the page text in steps, slice out at most one chunk's characters, and trim surrounding whitespace.

`if chunk_text:` and `chunks.append({` are used to add non-empty chunks with their ID, page, and text, then advance the ID.

`start += step_size` and `return chunks` are used to advance by the overlap-aware step and return all generated chunks.

`chunks` is used to run the function using the current parameter settings and save its result as chunks.



In [8]:
def create_chunks(
    pages,
    chunk_size,
    chunk_overlap
):
    """
    Split each page into character-based chunks.

    Chunking is performed independently for each page.
    Overlap is maintained between consecutive chunks
    from the same page.
    """

    chunks = []
    chunk_id = 0

    step_size = chunk_size - chunk_overlap

    for page_data in pages:

        page_number = page_data["page"]
        page_text = page_data["text"]

        start = 0

        while start < len(page_text):

            end = start + chunk_size

            chunk_text = page_text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "chunk_id": chunk_id,
                    "page": page_number,
                    "text": chunk_text
                })

                chunk_id += 1

            # Move forward while maintaining overlap
            start += step_size

    return chunks


chunks = create_chunks(
    pages,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

### 9. DISPLAY CHUNKING INFORMATION



### Cell 9: Display Chunking Information

This cell's purpose is to summarize the generated chunk set. These checks confirm chunk creation succeeded and show whether the chosen size produced the expected distribution before embedding begins. The last chunk on a page is often shorter than the chosen chunk size, which is expected.

`print("CHUNKING RESULTS")` is used to label the results section.

`print("Number of chunks:", len(chunks))` and `if len(chunks) == 0:` are used to display the chunk count and stop if no chunks were created.

`chunk_lengths = [` and `len(chunk["text"])` are used to measure each chunk's text length.

`round(np.mean(chunk_lengths), 2)` is used to calculate and display the mean chunk length.

`min(chunk_lengths),` and `max(chunk_lengths),` are used to show the shortest and longest chunk lengths to expose unusually small or large chunks.



In [9]:
print("\n")
print("=" * 70)
print("CHUNKING RESULTS")
print("=" * 70)

print("Number of chunks:", len(chunks))

if len(chunks) == 0:
    raise RuntimeError(
        "No chunks were generated. Check the extracted PDF text."
    )

chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print(
    "Average characters per chunk:",
    round(np.mean(chunk_lengths), 2)
)

print(
    "Shortest chunk:",
    min(chunk_lengths),
    "characters"
)

print(
    "Longest chunk:",
    max(chunk_lengths),
    "characters"
)



CHUNKING RESULTS
Number of chunks: 55
Average characters per chunk: 1105.25
Shortest chunk: 155 characters
Longest chunk: 1500 characters


### 10. EXPORT GENERATED CHUNKS TO EXCEL



### Cell 10: Export Generated Chunks To Excel

This cell's purpose is to export the generated chunks and their source metadata to Excel. The spreadsheet makes the exact retrieval units inspectable, supports manual relevance labeling, and helps validate page provenance and chunk settings. The exported workbook records exactly what the model can retrieve, which makes it useful for checking evidence and marking relevance.

`print("EXPORTING GENERATED CHUNKS TO EXCEL")` is used to label the export stage.

`chunk_records = []` and `for chunk in chunks:` are used to create one row per chunk, retaining its ID, source PDF, page, character count, chunk settings, and text.

`chunks_df = pd.DataFrame(chunk_records)` is used to convert the list of records into a pandas table.

`base_name` and `os.path.basename(pdf_file)` are used to build an output filename from the uploaded PDF's name.

`with pd.ExcelWriter` and `engine="openpyxl"` are used to create the Excel workbook and write the table to a named worksheet without an extra DataFrame index.

`worksheet = writer.sheets["Generated_Chunks"]` and `for cell in worksheet[1]:` are used to format header cells with bold white text, a dark fill, centered alignment, and wrapping.

`column_widths = {` and `for column, width in column_widths.items():` are used to set column widths so metadata and chunk text are easier to inspect.

`for row in worksheet.iter_rows` and `min_row=2,` are used to wrap long chunk text and align worksheet values at the top of each cell.

`worksheet.freeze_panes = "A2"` and `worksheet.auto_filter.ref = worksheet.dimensions` are used to freeze the header row, enable filtering, and set a readable header height.

`print("Excel file created:", EXCEL_FILE)` and `print("Chunks exported:", len(chunks_df))` are used to report the output filename, number of rows, and worksheet name.

`print("\nExcel export preview:")` and `chunks_df[` are used to print a compact preview of the first ten chunk IDs, pages, and lengths.

`files.download(EXCEL_FILE)` is used to trigger a download of the workbook in Google Colab.



In [10]:
print("\n")
print("=" * 70)
print("EXPORTING GENERATED CHUNKS TO EXCEL")
print("=" * 70)

# Prepare one Excel row for every generated chunk.
chunk_records = []

for chunk in chunks:

    chunk_records.append({
        "Chunk_ID": chunk["chunk_id"],
        "Source_Document": pdf_file,
        "Page_Number": chunk["page"],
        "Character_Count": len(chunk["text"]),
        "Chunk_Size_Setting": CHUNK_SIZE,
        "Chunk_Overlap_Setting": CHUNK_OVERLAP,
        "Chunk_Text": chunk["text"]
    })

chunks_df = pd.DataFrame(chunk_records)

# Create an output filename based on the source PDF.
base_name = os.path.splitext(
    os.path.basename(pdf_file)
)[0]

EXCEL_FILE = f"{base_name}_chunks_inspection.xlsx"

# Create and format the Excel workbook.
with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    chunks_df.to_excel(
        writer,
        sheet_name="Generated_Chunks",
        index=False
    )

    worksheet = writer.sheets["Generated_Chunks"]

    # Format the header row.
    for cell in worksheet[1]:

        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Set column widths.
    column_widths = {
        "A": 12,
        "B": 35,
        "C": 14,
        "D": 18,
        "E": 20,
        "F": 23,
        "G": 80
    }

    for column, width in column_widths.items():
        worksheet.column_dimensions[column].width = width

    # Format rows for easier inspection.
    for row in worksheet.iter_rows(
        min_row=2,
        max_row=worksheet.max_row
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=(cell.column == 7)
            )

    # Keep headers visible while scrolling.
    worksheet.freeze_panes = "A2"

    # Enable filtering.
    worksheet.auto_filter.ref = worksheet.dimensions

    # Set header height.
    worksheet.row_dimensions[1].height = 32

print("Excel file created:", EXCEL_FILE)
print("Chunks exported:", len(chunks_df))
print("Excel worksheet: Generated_Chunks")

print("\nExcel export preview:")

print(
    chunks_df[
        [
            "Chunk_ID",
            "Page_Number",
            "Character_Count"
        ]
    ].head(10).to_string(index=False)
)

# Automatically download the Excel workbook in Google Colab.
files.download(EXCEL_FILE)



EXPORTING GENERATED CHUNKS TO EXCEL
Excel file created: IT_and_Business_Analytics (1)_chunks_inspection.xlsx
Chunks exported: 55
Excel worksheet: Generated_Chunks

Excel export preview:
 Chunk_ID  Page_Number  Character_Count
        0            1             1500
        1            1              600
        2            2             1499
        3            2             1103
        4            3             1500
        5            3              445
        6            4             1355
        7            4              155
        8            5             1500
        9            5              634


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 11. DISPLAY SAMPLE CHUNKS

Print sample chunks to check that text boundaries and page metadata look sensible.

### Explanation — Cell 11: Display Sample Chunks

This cell's purpose is to display the first three chunks as a quick content check. A sample helps catch bad text extraction, incorrect page attribution, or awkward boundaries before vectors are created. This quick preview complements the full Excel inspection; it is useful for detecting obvious extraction errors early.

`print("SAMPLE CHUNKS")` is used to label the sample output.

`for chunk in chunks[:3]:` is used to select the first three chunk records.

`print(f"\nChunk ID : {chunk['chunk_id']}")` and `print(f"Page     : {chunk['page']}")` are used to show each sample's ID, source page, and character count.

`print(chunk["text"])` is used to separate and print the chunk text for direct inspection.



In [11]:
print("\n")
print("=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

for chunk in chunks[:3]:

    print(f"\nChunk ID : {chunk['chunk_id']}")
    print(f"Page     : {chunk['page']}")
    print(f"Characters: {len(chunk['text'])}")

    print("-" * 70)

    print(chunk["text"])



SAMPLE CHUNKS

Chunk ID : 0
Page     : 1
Characters: 1500
----------------------------------------------------------------------
Page 90 INFORMATION TECHNOLOGY PROGRAM The Bachelor of Science in Information Technology (BSIT) program highlights the practical application of information technology (IT). It provides students with the breadth and depth of IT needed for professional success in the field. It prepares students to be well-versed in application installation, operation, development, maintenance, administration, and familiarity with hardware installation, operation, and maintenance. PROGRAM EDUCATIONAL OBJECTIVES (PEO) Three to five years after completing the Bachelor of Science in Information Technology (BSIT), the graduates will: 1. excel in their profession/career utilizing the knowledge acquired in the Information Technology program; 2. become effective collaborators and innovators in information technology, applying professional/technical skills and competencies to make a p

### 12. LOAD EMBEDDING MODEL

Load the sentence-transformer used to encode both document chunks and future questions into the same vector space.

### Explanation — Cell 12: Load Embedding Model

This cell's purpose is to load the sentence embedding model. The same model will encode document chunks and user questions into vectors that can be compared during semantic retrieval. The embedding model represents meaning as numbers; it does not write the final chatbot answer.

`print("LOADING EMBEDDING MODEL")` is used to label the model-loading stage.

`embedding_model` is used to load the configured SentenceTransformer model into memory.

`print("Embedding model:", EMBEDDING_MODEL_NAME)` is used to print the model name to confirm which embedder is active.



In [12]:
print("\n")
print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)



LOADING EMBEDDING MODEL


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model: all-MiniLM-L6-v2


### 13. EXTRACT CHUNK TEXT



### Explanation — Cell 13: Extract Chunk Text

This cell's purpose is to create a text-only list from the chunk records. The embedding model accepts text strings; keeping this list in the same order as chunks preserves the mapping between vectors and source records. Changing the order here without changing the chunk records would attach the wrong page and text to a search result.

`chunk_texts = [` and `chunk["text"]` are used to build chunk_texts by taking the text field from each chunk without changing its order.



In [13]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

### 14. CREATE DOCUMENT EMBEDDINGS



### Cell 14: Create Document Embeddings

This cell's purpose is to encode every chunk as a normalized vector. FAISS needs numeric vectors to perform similarity search. Normalization makes inner-product scores comparable to cosine similarity. One row of the resulting matrix corresponds to one source chunk.

`print("CREATING DOCUMENT EMBEDDINGS")` is used to label the embedding step.

`embeddings` and `convert_to_numpy=True,` are used to encode the ordered chunk texts, return NumPy arrays, normalize each vector, and show progress.

`embeddings = embeddings.astype("float32")` is used to convert vectors to float32, the data type expected by the FAISS index used below.

`print("\nEmbedding matrix shape:")` and `print(embeddings.shape)` are used to report the embedding matrix dimensions for verification.



In [14]:
print("\n")
print("=" * 70)
print("CREATING DOCUMENT EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# FAISS expects float32 vectors.
embeddings = embeddings.astype("float32")

print("\nEmbedding matrix shape:")
print(embeddings.shape)



CREATING DOCUMENT EMBEDDINGS


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Embedding matrix shape:
(55, 384)


### 15. CREATE FAISS VECTOR STORE


### Cell 15: Create FAISS Vector Store

This cell's purpose is to build the FAISS vector index from the document embeddings. The index is the searchable representation of the corpus; its row order must stay aligned with the chunk list for correct source recovery. The number of stored vectors should equal the number of chunks created earlier.

`print("CREATING FAISS VECTOR STORE")` is used to label vector-store construction.

`embedding_dimension = embeddings.shape[1]` is used to read the vector width from the embedding matrix.

`index` is used to create an inner-product index; because vectors were normalized, scores correspond to cosine similarity.

`index.add(embeddings)` is used to add all document vectors to the index in their existing order.

`print("Embedding dimension:", embedding_dimension)` and `print("Vectors stored:", index.ntotal)` are used to print the vector dimension and indexed vector count.



In [15]:
print("\n")
print("=" * 70)
print("CREATING FAISS VECTOR STORE")
print("=" * 70)

embedding_dimension = embeddings.shape[1]

# Inner Product on normalized vectors is equivalent
# to cosine similarity.
index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors stored:", index.ntotal)



CREATING FAISS VECTOR STORE
Embedding dimension: 384
Vectors stored: 55


### 16. CREATE CHUNK LOOKUP



### Cell 16: Create Chunk Lookup

This cell's purpose is to map each FAISS row position back to its original chunk record. FAISS returns vector positions rather than page and chunk metadata. This lookup is what lets retrieval return the correct text and citation details. If this cell is skipped, retrieval cannot resolve FAISS positions to chunks and may raise a missing-variable error.

`# The embedding row index corresponds to the position` and `# of the chunk in the chunks list.` are used to document the required alignment between vector positions and the chunks list.

`chunk_lookup = {` and `position: chunk` are used to create a dictionary whose keys are zero-based embedding positions and whose values are the matching chunk records.



In [16]:
# The embedding row index corresponds to the position
# of the chunk in the chunks list.

chunk_lookup = {
    position: chunk
    for position, chunk in enumerate(chunks)
}

### 17. DEFINE RETRIEVAL FUNCTION



### Cell 17: Define Retrieval Function

This cell's purpose is to define semantic retrieval for one user question. This function is the bridge from a question to ranked source passages; it returns evidence with IDs, pages, and scores for later prompting and evaluation. The returned source details allow the chatbot answer to be checked against the original PDF.

`retrieve_documents` and `top_k=3` are used to define the query and Top K inputs and document the fields returned.

`if not query or not query.strip():` and `raise ValueError("The query cannot be empty.")` are used to reject blank questions and non-positive retrieval counts.

`query_embedding` and `[query],` are used to encode the question with the same normalized embedding model and cast it to float32.

`search_k = min(top_k, index.ntotal)` and `scores, indices` are used to limit search depth to the number of indexed chunks and ask FAISS for the closest vectors.

`results = []` and `for rank, idx in enumerate(indices[0]):` are used to walk through ranked results, skip empty FAISS slots, and resolve each position through chunk_lookup.

`results.append({` and `})` are used to return rank, source chunk ID, page, similarity score, and text for each result.

`return results` is used to return the complete ranked result list.



In [17]:
def retrieve_documents(
    query,
    top_k=3
):
    """
    Retrieve the top matching chunks using FAISS.

    Returns chunk ID, page, similarity score, and text.
    """

    if not query or not query.strip():
        raise ValueError("The query cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Convert the question into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    # Avoid requesting more results than available vectors.
    search_k = min(top_k, index.ntotal)

    # Search the FAISS vector store.
    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        if idx == -1:
            continue

        chunk = chunk_lookup[int(idx)]

        results.append({
            "rank": rank + 1,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(scores[0][rank]),
            "text": chunk["text"]
        })

    return results

### 18. TEST RETRIEVAL



### Cell 18: Test Retrieval

This cell's purpose is to run a manual retrieval test before the LLM is involved. This isolates embedding and FAISS behavior so retrieval quality can be checked independently of answer generation. A poor result here points to chunking or retrieval, before the language model is involved.

`print("TESTING VECTOR STORE")` is used to label the vector-store test.

`test_query` is used to ask for a test question, allowing Enter to skip this optional test.

`if test_query.strip():` and `test_results` handle a conditional case. If a question was entered, retrieve up to the configured Top K results.

`print("RETRIEVED CHUNKS")` is used to add a heading for the returned passages.

`for result in test_results:` and `print(f"\nRESULT {result['rank']}")` are used to print each result's rank, chunk, page, similarity, and text so relevance can be judged.



In [18]:
print("\n")
print("=" * 70)
print("TESTING VECTOR STORE")
print("=" * 70)

test_query = input(
    "\nEnter a test question "
    "(or press Enter to skip): "
)

if test_query.strip():

    test_results = retrieve_documents(
        test_query,
        TOP_K
    )

    print("\n")
    print("=" * 70)
    print("RETRIEVED CHUNKS")
    print("=" * 70)

    for result in test_results:

        print(f"\nRESULT {result['rank']}")
        print(f"Chunk ID   : {result['chunk_id']}")
        print(f"Page       : {result['page']}")
        print(f"Similarity : {result['score']:.4f}")

        print("-" * 70)

        print(result["text"])



TESTING VECTOR STORE

Enter a test question (or press Enter to skip): 


### 19. SAVE FAISS INDEX



### Cell 19: Save FAISS Index

This cell's purpose is to save the FAISS index to disk. Saving avoids rebuilding the vector index when the same chunk set is reused later. The saved index must be paired with the matching chunk metadata from the same run.

`FAISS_FILE = "rag_faiss.index"` is used to choose a stable filename for the index.

`faiss.write_index` is used to serialize the in-memory FAISS index to that file.

`print("\nFAISS index saved:", FAISS_FILE)` is used to confirm the saved index path.



In [19]:
FAISS_FILE = "rag_faiss.index"

faiss.write_index(
    index,
    FAISS_FILE
)

print("\nFAISS index saved:", FAISS_FILE)


FAISS index saved: rag_faiss.index


### 20. SAVE CHUNKS AND METADATA



### Cell 20: Save Chunks And Metadata

This cell's purpose is to save the original chunk records alongside the index. The vector index contains vectors but not the complete source text and page metadata; this file preserves the records needed to interpret search results. The order of these saved chunks must match the order used when vectors were added to FAISS.

`CHUNKS_FILE = "rag_chunks.pkl"` is used to choose a filename for the chunk metadata.

`with open(CHUNKS_FILE, "wb") as f:` and `pickle.dump` are used to open the file in binary write mode and serialize the chunks list with pickle.

`print("Chunks saved:", CHUNKS_FILE)` is used to confirm the saved chunk file.



In [20]:
CHUNKS_FILE = "rag_chunks.pkl"

with open(CHUNKS_FILE, "wb") as f:

    pickle.dump(
        chunks,
        f
    )

print("Chunks saved:", CHUNKS_FILE)

Chunks saved: rag_chunks.pkl


### 21. SAVE INDEXING CONFIGURATION



### Cell 21: Save Indexing Configuration

This cell's purpose is to save the settings used to build the index. Recording the parameters makes runs reproducible and helps identify which chunking, embedding, and retrieval setup produced a result. These values allow the group to identify and compare a specific experimental configuration later.

`CONFIG_FILE = "rag_config.pkl"` is used to choose a filename for the configuration.

`config = {` is used to collect chunk size, overlap, Top K, embedding model, and source document into one dictionary.

`with open(CONFIG_FILE, "wb") as f:` and `pickle.dump` are used to serialize the configuration dictionary to disk.

`print("Configuration saved:", CONFIG_FILE)` is used to confirm the saved configuration path.



In [21]:
CONFIG_FILE = "rag_config.pkl"

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "source_document": pdf_file
}

with open(CONFIG_FILE, "wb") as f:

    pickle.dump(
        config,
        f
    )

print("Configuration saved:", CONFIG_FILE)

Configuration saved: rag_config.pkl


### 22. INDEXING COMPLETE



### Cell 22: Indexing Complete

This cell's purpose is to summarize the completed indexing run and its files. This final checkpoint confirms that chunks and vectors agree and tells the user where each artifact was saved before moving to the answering component. This report is a final check that the expected files and counts exist before answering questions.

`print("INDEXING COMPLETE")` is used to label successful completion of indexing.

`print("Source document :", pdf_file)` and `print("Number of chunks:", len(chunks))` are used to report source, chunk count, indexed vector count, chunk settings, and embedding model.

`print("\nSaved files:")` and `print("  ", EXCEL_FILE)` are used to list the Excel inspection workbook, FAISS index, chunk records, and settings file.

`print("\nThe vector store is ready for retrieval.")` is used to state that retrieval is ready and remind the user to inspect the exported chunks.



In [22]:
print("\n")
print("=" * 70)
print("INDEXING COMPLETE")
print("=" * 70)

print("Source document :", pdf_file)
print("Number of chunks:", len(chunks))
print("Vectors stored  :", index.ntotal)
print("Chunk size      :", CHUNK_SIZE)
print("Chunk overlap   :", CHUNK_OVERLAP)
print("Embedding model :", EMBEDDING_MODEL_NAME)

print("\nSaved files:")
print("  ", EXCEL_FILE)
print("  ", FAISS_FILE)
print("  ", CHUNKS_FILE)
print("  ", CONFIG_FILE)

print("\nThe vector store is ready for retrieval.")
print(
    "Inspect the downloaded Excel workbook to evaluate "
    "the generated chunks."
)



INDEXING COMPLETE
Source document : IT_and_Business_Analytics (1).pdf
Number of chunks: 55
Vectors stored  : 55
Chunk size      : 1500
Chunk overlap   : 300
Embedding model : all-MiniLM-L6-v2

Saved files:
   IT_and_Business_Analytics (1)_chunks_inspection.xlsx
   rag_faiss.index
   rag_chunks.pkl
   rag_config.pkl

The vector store is ready for retrieval.
Inspect the downloaded Excel workbook to evaluate the generated chunks.


# Component II: Prompt and Retrieval



### 1. IMPORT REQUIRED LIBRARIES



### Cell 1: Import Required Libraries

This cell's purpose is to import the libraries needed for prompt construction and answer generation. These dependencies provide model and tokenizer loading, 4-bit quantization, tensor operations, and regular-expression cleanup. The tokenizer and model loader come from Transformers, while PyTorch supplies tensor and device operations.

`from transformers import ` is used to import the Hugging Face tokenizer, causal-language-model loader, and bitsandbytes quantization configuration.

`import torch` and `import re` are used to import PyTorch for device and generation operations and re for text cleanup.



In [23]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch
import re

### 2. CHECK DEVICE



### Cell 2: Check Device

This cell's purpose is to detect whether a CUDA GPU is available. The notebook's 4-bit Qwen3-8B loading path is configured for a GPU; reporting the device early makes runtime requirements visible. The cell reports CPU when no CUDA GPU is available, but the later 4-bit loading cell requires a GPU in this notebook.

`device` and `if torch.cuda.is_available()` are used to choose cuda when PyTorch detects a GPU; otherwise label the device cpu.

`print("RAG PROMPT PROCESSING")` and `print("Device:", device)` are used to print the prompt-processing stage and the selected device.



In [24]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("RAG PROMPT PROCESSING")
print("=" * 70)

print("Device:", device)

RAG PROMPT PROCESSING
Device: cuda


### 3. LOAD THE LOCAL LLM



### Cell 3: Load The Local LLM

This cell's purpose is to load Qwen3-8B in 4-bit mode and set token limits. Quantization reduces GPU memory use, while the token budgets constrain prompt size and answer length. This notebook cell intentionally stops on CPU because this configured model-loading path is GPU-oriented. Model loading can take time because the weights must be downloaded and placed in memory; the notebook then uses the loaded model for all later questions.

`LLM_NAME = "Qwen/Qwen3-8B"` and `MAX_INPUT_TOKENS = 4096` are used to set the model ID, maximum prompt tokens, and maximum generated answer tokens.

`RESERVED_TOKEN_MARGIN = 64` is used to reserve a safety margin so chat formatting does not push the prompt past the model input budget.

`print("\nLoading language model:")` and `print(LLM_NAME)` are used to print which model is about to load.

`if not torch.cuda.is_available():` is used to fail early with a clear message if no CUDA GPU is available for this notebook configuration.

`compute_dtype` and `torch.bfloat16` are used to select a GPU compute type supported by the current hardware.

`quantization_config` and `load_in_4bit=True,` are used to configure 4-bit NF4 weights and double quantization using the selected compute type.

`tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)` is used to load the matching tokenizer.

`llm` and `quantization_config=quantization_config,` are used to load Qwen3-8B with the quantization settings, the current compute dtype, and automatic device placement.

`llm.eval()` and `device = llm.get_input_embeddings().weight.device` are used to switch the model to evaluation mode, record its input device, and confirm loading.



In [25]:
LLM_NAME = "Qwen/Qwen3-8B"

MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 256

# Reserve space within the input budget for safety.
RESERVED_TOKEN_MARGIN = 64

print("\nLoading language model:")
print(LLM_NAME)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Qwen3-8B needs a GPU runtime for this notebook. "
        "In Google Colab, select Runtime > Change runtime type > T4 GPU, then rerun."
    )

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=quantization_config,
    dtype=compute_dtype,
    device_map="auto"
)
llm.eval()
device = llm.get_input_embeddings().weight.device

print("Language model loaded.")


Loading language model:
Qwen/Qwen3-8B


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Language model loaded.


### 4. DEFINE TOKEN COUNTING FUNCTION



### Cell 4: Define Token Counting Function

This cell's purpose is to define a helper that counts tokens with the loaded tokenizer. Token counts, rather than character counts, determine whether text fits within the language model's context limit. A sentence's character count and token count can differ greatly, so the model's limit must be checked in tokens.

`def count_tokens(text):` is used to define the helper for any input text.

`return len` and `tokenizer.encode` are used to tokenize without adding special tokens or truncating, then return the number of token IDs.



In [26]:
def count_tokens(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False,
            truncation=False
        )
    )

### 5. PRIORITIZE RETRIEVED DOCUMENTS



### Cell 5: Prioritize Retrieved Documents

This cell's purpose is to keep retrieved documents ordered by similarity score. The prompt builder should use the strongest evidence first so the most relevant passages are less likely to be excluded by the token budget. Preserving the search order is especially useful when the prompt cannot fit all retrieved passages.

`def prioritize_documents(question, retrieved_documents):` is used to define the ranking helper and document that higher inner-product similarity means a closer match.

`return sorted` and `key=lambda result: result.get("score", float("-inf")),` are used to sort descending by score, treating a missing score as very low, and return the ordered results.



In [27]:
def prioritize_documents(question, retrieved_documents):
    """
    Preserve FAISS similarity ranking.

    Assumes higher scores mean greater similarity,
    as with normalized embeddings and IndexFlatIP.
    """

    return sorted(
        retrieved_documents,
        key=lambda result: result.get("score", float("-inf")),
        reverse=True
    )

### 6. DEFINE THE TOKEN-LIMITED RAG PROMPT



### ECell 6: Define The Token-Limited RAG Prompt

This cell's purpose is to build a grounded prompt while respecting the model's input-token budget. This is the core grounding step: it keeps the inquiry and source evidence together, prioritizes better matches, and prevents silent truncation from removing the question or citations. Page and chunk labels remain attached to the evidence, making the final prompt easier to audit.

`create_rag_prompt` is used to define the inputs and state that evidence is trimmed only at token boundaries.

`ranked_documents` is used to order retrieved chunks by relevance before allocating the context budget.

`#instruction = (` and `#    "Answer the question using only the evidence below. "` are inactive. The notebook keeps these older instructions for reference, but they do not affect the current prompt.

`instruction` is used to set the active instruction: answer only from JRU evidence, keep BSIT and BSIT-BA distinct, avoid unsupported advice, and use the specified fallback when evidence is insufficient.

`prompt_prefix` and `f"Question: {question}\n\n"` are used to place the question and instructions before an Evidence section and reserve an Answer suffix.

`prefix_tokens` and `add_special_tokens=False` are used to tokenize the fixed prompt prefix and suffix so their cost can be subtracted first.

`available_context_tokens` and `- len(prefix_tokens)` are used to calculate the remaining evidence budget, including a safety margin, and stop with a useful error if no evidence can fit.

`selected_parts = []` and `remaining_tokens = available_context_tokens` are used to initialize selected evidence and attach page and chunk labels to each source passage.

`source_tokens` and `add_special_tokens=False` are used to tokenize each passage and account for separators between passages.

`included_tokens = source_tokens[` and `:tokens_available` are used to include a whole passage when possible; otherwise truncate at a token boundary and stop adding later passages.

`context = "\n\n".join(selected_parts)` and `prompt_text` are used to join selected passages and assemble the user-facing prompt text.

`prompt` and `[{"role": "user", "content": prompt_text}],` are used to apply Qwen3's chat template and disable its thinking mode for concise inquiry answers.

`final_token_count = count_tokens(prompt)` and `if final_token_count > MAX_INPUT_TOKENS:` are used to count the fully formatted prompt and raise an error if it exceeds the configured input limit.

`print("\n--- PROMPT DIAGNOSTICS ---")` and `print("Original question:", question)` are used to print token-budget diagnostics for troubleshooting.

`print("\n--- SELECTED EVIDENCE ---")` and `for i, part in enumerate(selected_parts, start=1):` are used to display the evidence actually included and return the final model-ready prompt.



In [28]:
def create_rag_prompt(
    question,
    retrieved_documents
):
    """
    Build a prompt that preserves the question and includes
    as much relevant retrieved evidence as the input budget
    allows.

    Context is trimmed only at token boundaries.
    """

    ranked_documents = prioritize_documents(
        question,
        retrieved_documents
    )

    #instruction = (
    #    "Answer the question using only the evidence below. "
    #    "Give a concise answer. Do not invent information. "
    #    'If the evidence does not support an answer, say: '
    #    '"I cannot find the answer in the provided document."'
    #)

    instruction = (
        "Answer using only the supplied José Rizal University evidence. The inquiry scope "
        "is limited to the Bachelor of Science in Information Technology (BSIT) and BSIT "
        "with Specialization in Business Analytics (BSIT-BA). Explain only details supported "
        "by the evidence, such as program descriptions, subjects, curriculum structure, "
        "course descriptions, and stated prerequisites. Keep program names, course codes, "
        "and school-year labels accurate. Clearly distinguish BSIT from BSIT-BA. The source "
        "may describe electives for other IT tracks; do not present those as required BSIT "
        "or BSIT-BA courses unless the curriculum evidence explicitly says so. Do not invent "
        "facts, use outside knowledge, make admission-eligibility decisions, or claim a "
        "program is personally suitable for the user. If the evidence does not answer the "
        "question, respond exactly: "
        "'I cannot find the answer in the provided document.'"
    )

    # Keep the question near the beginning.
    prompt_prefix = (
        f"Question: {question}\n\n"
        f"{instruction}\n\n"
        "Evidence:\n"
    )

    prompt_suffix = "\n\nAnswer:"

    # Tokenize the fixed parts first.
    prefix_tokens = tokenizer.encode(
        prompt_prefix,
        add_special_tokens=False
    )

    suffix_tokens = tokenizer.encode(
        prompt_suffix,
        add_special_tokens=False
    )

    # Keep the final prompt safely within the model limit.
    available_context_tokens = (
        MAX_INPUT_TOKENS
        - len(prefix_tokens)
        - len(suffix_tokens)
        - RESERVED_TOKEN_MARGIN
        - 1  # Account for a possible special token.
    )

    if available_context_tokens <= 0:
        raise ValueError(
            "The question and instructions leave no room "
            "for retrieved evidence. Shorten the question."
        )

    selected_parts = []
    remaining_tokens = available_context_tokens

    for result in ranked_documents:

        if remaining_tokens <= 0:
            break

        source_text = (
            f"[Page {result['page']}, "
            f"Chunk {result['chunk_id']}]\n"
            f"{result['text']}"
        )

        # Tokenize each source separately.
        source_tokens = tokenizer.encode(
            source_text,
            add_special_tokens=False
        )

        separator_tokens = (
            tokenizer.encode(
                "\n\n",
                add_special_tokens=False
            )
            if selected_parts
            else []
        )

        tokens_available = (
            remaining_tokens - len(separator_tokens)
        )

        if tokens_available <= 0:
            break

        # Include the full chunk when it fits.
        # Otherwise, retain the beginning of the chunk.
        included_tokens = source_tokens[
            :tokens_available
        ]

        if not included_tokens:
            continue

        included_text = tokenizer.decode(
            included_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        selected_parts.append(included_text)

        remaining_tokens -= (
            len(separator_tokens)
            + len(included_tokens)
        )

        # Stop if this chunk had to be truncated.
        if len(included_tokens) < len(source_tokens):
            break

    context = "\n\n".join(selected_parts)

    prompt_text = (
        prompt_prefix
        + context
        + prompt_suffix
    )

    # Format the user prompt with Qwen3's chat template in non-thinking mode.
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    # Final check counts the full chat-formatted prompt.
    final_token_count = count_tokens(prompt)

    if final_token_count > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Prompt contains {final_token_count} tokens, "
            f"exceeding the {MAX_INPUT_TOKENS}-token limit. "
            "Reduce the context budget."
        )

    print("\n--- PROMPT DIAGNOSTICS ---")
    print("Original question:", question)
    print("Retrieved chunks:", len(retrieved_documents))
    print("Chunks included:", len(selected_parts))
    print("Final prompt tokens:", final_token_count)
    print("Input token limit:", MAX_INPUT_TOKENS)

    print("\n--- SELECTED EVIDENCE ---")

    for i, part in enumerate(selected_parts, start=1):
      print(f"\nEvidence {i}:")
      print(part)

    return prompt

### 7. DEFINE ANSWER GENERATION FUNCTION



### Cell 7: Define Answer Generation Function

This cell's purpose is to generate an answer from the grounded prompt. This function checks the actual input length, avoids backpropagation during inference, and decodes only newly generated answer tokens. A causal model returns the input prompt followed by new tokens; slicing removes the repeated prompt from the displayed answer.

`generate_answer` and `max_new_tokens=MAX_NEW_TOKENS` are used to define the generation helper and document its no-silent-truncation behavior.

`inputs` and `return_tensors="pt",` are used to tokenize the prompt into tensors without truncation or extra special tokens.

`input_length = inputs["input_ids"].shape[1]` and `print("\n--- GENERATION DIAGNOSTICS ---")` are used to measure input length, print diagnostics, and reject prompts over the configured limit.

`inputs = {` and `key: value.to(device)` are used to move tokenizer tensors to the device used by the loaded model.

`with torch.no_grad():` and `outputs` are used to disable gradient tracking and generate a deterministic response with configured length and stop tokens.

`generated_tokens = outputs[0, input_length:]` and `answer` are used to remove the prompt portion from the output and decode only the model's new tokens.

`if not answer:` and `return answer` are used to return a helpful message for empty output; otherwise return the answer text.



In [29]:
def generate_answer(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS
):
    """
    Generate an answer without silently truncating the prompt.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=False
    )

    input_length = inputs["input_ids"].shape[1]

    print("\n--- GENERATION DIAGNOSTICS ---")
    print("Actual input tokens:", input_length)
    print("Maximum input tokens:", MAX_INPUT_TOKENS)

    if input_length > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Input contains {input_length} tokens, "
            f"but the model limit is {MAX_INPUT_TOKENS}."
        )

    # Move model inputs to the selected device.
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate deterministically.
    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Causal language models return the prompt followed by the new answer.
    generated_tokens = outputs[0, input_length:]
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        return (
            "The model generated an empty answer. "
            "Inspect the prompt and retrieved evidence."
        )

    return answer

### 8. DEFINE COMPLETE RAG QUERY FUNCTION



### Cell 8: Define Complete RAG Query Function

This cell's purpose is to connect retrieval, evidence selection, and generation into one RAG function. A single function gives the interactive loop a consistent workflow and a predictable return value for each inquiry. The function returns the answer together with the sources and prompt, so the interactive interface can show how it was produced.

`def rag_query(question, top_k=3):` is used to define the function and document the four-stage RAG sequence and returned values.

`question = question.strip()` and `if not question:` are used to trim and validate the question; return a friendly prompt instead of proceeding with empty input.

`retrieved_documents` and `top_k=top_k` are used to retrieve the most relevant chunks using the requested Top K.

`if not retrieved_documents:` and `[],` are used to return the grounded fallback when retrieval finds no evidence.

`prompt` is used to build a token-limited prompt from the question and retrieved sources.

`answer = generate_answer(prompt)` and `if not answer or not answer.strip():` are used to generate an answer and replace an empty response with a useful diagnostic message.

`answer.strip(),` is used to return the answer, source records, and exact prompt for display or inspection.



In [30]:
def rag_query(question, top_k=3):
    """
    General-purpose RAG question-answering function.

    Workflow:
    1. Retrieve relevant document chunks.
    2. Build a token-limited prompt.
    3. Generate an answer using the language model.
    4. Return the answer, retrieved sources, and prompt.
    """

    # STEP 1: Validate the question.
    question = question.strip()

    if not question:
        return (
            "Please enter a valid question.",
            [],
            ""
        )

    # STEP 2: Retrieve relevant document chunks.
    retrieved_documents = retrieve_documents(
        question,
        top_k=top_k
    )

    # STEP 3: Handle cases where no evidence is retrieved.
    if not retrieved_documents:
        return (
            "I cannot find the answer in the provided document.",
            [],
            ""
        )

    # STEP 4: Construct the token-limited RAG prompt.
    prompt = create_rag_prompt(
        question,
        retrieved_documents
    )

    # STEP 5: Generate an answer from the retrieved evidence.
    answer = generate_answer(prompt)

    # STEP 6: Handle an empty model response.
    if not answer or not answer.strip():
        answer = (
            "The model could not generate an answer. "
            "Please review the retrieved evidence."
        )

    # STEP 7: Return the results.
    return (
        answer.strip(),
        retrieved_documents,
        prompt
    )

### 9. CONTINUOUS QUESTION-ANSWER LOOP



### Cell 9: Continuous Question-Answer Loop

This cell's purpose is to run a continuous question-and-answer session for the indexed document. The loop lets users ask multiple inquiries, inspect the evidence and prompt used for the latest answer, and exit without rerunning model setup. The special commands let a user inspect the last answer's evidence without generating another answer.

`print("CONTINUOUS RAG QUESTION ANSWERING")` and `print("Ask questions about the indexed document.")` are used to print usage instructions, including the exit, sources, and prompt commands.

`last_answer = None` and `last_sources = []` are used to initialize storage for the latest answer, retrieved sources, and prompt.

`while True:` and `user_question` are used to repeat until exit and read each user's question.

`if user_question.lower() == "exit":` and `print("\nRAG session terminated.")` are used to handle the exit command and terminate the session cleanly.

`if not user_question:` and `print("Please enter a question.")` are used to ignore blank input and return to the prompt for a valid question.

`if user_question.lower() == "sources":` and `if not last_sources:` are used to implement the sources command, including a no-history check and display of ranks, pages, chunk IDs, scores, and text.

`if user_question.lower() == "prompt":` and `if last_prompt is None:` are used to implement the prompt command and show the last prompt if one exists.

`answer, retrieved_documents, prompt` and `top_k=TOP_K` handle ordinary user questions. For ordinary inquiries, call the RAG pipeline and save the latest answer, evidence, and prompt.

`print("RAG ANSWER")` and `print(answer)` are used to display the generated answer.

`print("RETRIEVED SOURCES")` and `for result in retrieved_documents:` are used to display each retrieved source with rank, page, chunk ID, score, and text.

`print("RAG PROMPT SENT TO LLM")` and `print(prompt)` are used to display the exact prompt sent to the language model for transparency.

`except Exception as e:` and `print("\nERROR:")` are used to catch query-time errors, show the error message, and keep the failure understandable.



In [31]:
print("\n")
print("=" * 70)
print("CONTINUOUS RAG QUESTION ANSWERING")
print("=" * 70)

print("Ask questions about the indexed document.")
print("Type 'exit' to terminate.")
print("Type 'sources' to display sources from the last query.")
print("Type 'prompt' to display the last generated prompt.")


# Store the most recent query results.
last_answer = None
last_sources = []
last_prompt = None


while True:

    print("\n")

    user_question = input(
        "Question: "
    ).strip()

    # --------------------------------------------------------
    # EXIT CONDITION
    # --------------------------------------------------------

    if user_question.lower() == "exit":

        print("\nRAG session terminated.")
        break

    # --------------------------------------------------------
    # IGNORE EMPTY QUESTIONS
    # --------------------------------------------------------

    if not user_question:

        print("Please enter a question.")
        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: SOURCES
    # --------------------------------------------------------

    if user_question.lower() == "sources":

        if not last_sources:
            print("No previous query sources are available.")
            continue

        print("\n" + "=" * 70)
        print("SOURCES FROM LAST QUERY")
        print("=" * 70)

        for result in last_sources:

            print(
                f"\nRank {result['rank']} | "
                f"Page {result['page']} | "
                f"Chunk {result['chunk_id']} | "
                f"Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: PROMPT
    # --------------------------------------------------------

    if user_question.lower() == "prompt":

        if last_prompt is None:
            print("No previous query prompt is available.")
            continue

        print("\n" + "=" * 70)
        print("LAST RAG PROMPT")
        print("=" * 70)

        print(last_prompt)

        continue

    # --------------------------------------------------------
    # PROCESS QUESTION
    # --------------------------------------------------------

    try:

        answer, retrieved_documents, prompt = rag_query(
            user_question,
            top_k=TOP_K
        )

        last_answer = answer
        last_sources = retrieved_documents
        last_prompt = prompt

        # ----------------------------------------------------
        # DISPLAY ANSWER
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG ANSWER")
        print("=" * 70)

        print(answer)

        # ----------------------------------------------------
        # DISPLAY RETRIEVED SOURCES
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RETRIEVED SOURCES")
        print("=" * 70)

        for result in retrieved_documents:

            print(
                f"\n[{result['rank']}] "
                f"Page {result['page']} "
                f"| Chunk {result['chunk_id']} "
                f"| Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        # ----------------------------------------------------
        # DISPLAY PROMPT
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG PROMPT SENT TO LLM")
        print("=" * 70)

        print(prompt)

    except Exception as e:

        print("\nERROR:")
        print(str(e))

        print("\nThe question could not be processed.")



CONTINUOUS RAG QUESTION ANSWERING
Ask questions about the indexed document.
Type 'exit' to terminate.
Type 'sources' to display sources from the last query.
Type 'prompt' to display the last generated prompt.


Question: How many total units does the BSIT curriculum list?

--- PROMPT DIAGNOSTICS ---
Original question: How many total units does the BSIT curriculum list?
Retrieved chunks: 3
Chunks included: 3
Final prompt tokens: 1252
Input token limit: 4096

--- SELECTED EVIDENCE ---

Evidence 1:
[Page 3, Chunk 4]
Page 92 THIRD YEAR First Semester Units Second Semester Units ITC C309 Presentation Skills for IT 3 CPE C204 Technopreneurship 101 3 ITC C311 Technical Writing for IT 3 ITC C302 System Integrated & Arch 1 3 ITC C313 Project Management 3 ITC C306 Network Technologies 2 3 ITC C305 Network Technologies 1 3 ITC C304 Information Security 2 3 ITC C303 Information Security 1 3 ITC C310 IT Project 1 3 ITC C301 Integrated Programming &Tech 1 3 ITC C506 ITE Elective 3 3 ITC C503 ITE E

### 10. END



### Cell 10: End

This cell's purpose is to mark the end of the interactive RAG program. This final cell gives a clear completion message when the user exits the continuous question-answer loop. This closing message appears only after the user enters the exit command.

`print("RAG PROGRAM FINISHED")` is used to print spacing and a closing banner after the loop has ended.



In [32]:
print("\n")
print("=" * 70)
print("RAG PROGRAM FINISHED")
print("=" * 70)



RAG PROGRAM FINISHED
